[Reference](https://medium.com/@rahul.kumar0/langgraph-persistence-the-feature-that-makes-everything-else-possible-d12d2805b983$0)

In [1]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict

llm = ChatOpenAI()

class JokeState(TypedDict):
    topic: str
    joke: str
    explanation: str

def generate_joke(state: JokeState):
    response = llm.invoke(f"Generate a joke on the topic: {state['topic']}")
    return {"joke": response.content}

def generate_explanation(state: JokeState):
    response = llm.invoke(f"Explain this joke: {state['joke']}")
    return {"explanation": response.content}

# Build graph
graph = StateGraph(JokeState)
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)
graph.add_edge("__start__", "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", "__end__")

# Add persistence - this is the key line
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [2]:
config = {"configurable": {"thread_id": "1"}}

result = workflow.invoke(
    {"topic": "pizza"},
    config=config
)

# Get the final state
workflow.get_state(config)

# Get ALL intermediate states
workflow.get_state_history(config)

config_2 = {"configurable": {"thread_id": "2"}}
result_2 = workflow.invoke({"topic": "pasta"}, config=config_2)